# 

In [ ]:
import sys
import ast
import numpy as np
import pickle
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting
import pandas as pd
import os
import fnmatch


input_directories = ["/rdata/ian/pico/paperRuns/pinsga2_oneoffs/pinsga_25to30_year_2000_2024-07-24_14-52-06"]

input_directories = [directory.rstrip('/') for directory in input_directories]

output = "/rdata/ian/pico/paperRuns"

raw_master_table = {'year':[], 'yield':[], 'irr_total':[], 'front':[], 'irrigation':[], 'run':[], 'gen':[], 'algorithm':[]}

In [ ]:
def parse_directory_name(full_path):

    file_name = full_path.split("/")[-1]
    
    fields = file_name.split("_")

    result = {}
    
    result["algorithm"] = fields[0]
    result["DM_range"] = fields[1]
    result["year"] = int(fields[3])
    result["run_date"] = fields[4]
    result["run_time"] = fields[4]

    return result

    

In [ ]:
def parse_file_name(file_name): 

    results = {}

    file_chunks = file_name.split("_")
    results["run"] = int(file_chunks[0][3:9])
    results["gen"] = int(file_chunks[1][3:9])

    return results 
    

In [ ]:
def get_file_names(directory, pattern): 
    
    matching_files = []
    for filename in os.listdir(directory):
        if fnmatch.fnmatch(filename, pattern):
            matching_files.append(os.path.join(directory, filename))

    return matching_files

In [ ]:
def read_files(files):

    all_solutions = None
    
    for (i, file_path) in enumerate(files):

        # Gather info from the file name
        file_name = file_path.split("/")[-1]
        full_dir = "/".join(file_path.split("/")[:-1])

        run_meta = parse_directory_name(full_dir)

        run_meta.update(parse_file_name(file_name) )
        
        # Get objcetive data
        current_objs = pd.read_csv(file_path, delimiter=',', names=["yield", "leaching"])

        current_objs["yield"] = current_objs["yield"] * -1
        current_objs["run"] = run_meta["run"]
        current_objs["gen"] = run_meta["gen"]

        # Get decision variable data 
        var_file_path = file_path[:-7] + "var.csv"
        current_vars = pd.read_csv(var_file_path, delimiter=',', header=None)

        headers = ["var%s" % header for header in range(current_vars.shape[1])]
        current_vars = current_vars.set_axis(headers, axis=1)
        
        current_objs = pd.concat([current_objs,current_vars], axis=1)
        
        if all_solutions is None:
            all_solutions = current_objs
        else:
            all_solutions = pd.concat([all_solutions, current_objs])

    return all_solutions


In [ ]:
objs = None

for directory in input_directories:


    dir_meta = parse_directory_name(directory)

    year = dir_meta["year"]
        
    print("Processing year %s for folder %s" % (year, directory))

    obj_files = get_file_names(directory, 'run*obj.csv')

    objs = read_files(obj_files)
    
objs


